# 🐍 Join Datasets Notebook Overview
This notebook processes and merges IMDb and TMDB datasets, applying filtering and cleanup.

## 📂 Steps in This Notebook
1️⃣ **Unpacked IMDb Data** → Extracted GZIP files (one-time setup).  
2️⃣ **Loaded IMDb Title Basics** → Dropped unnecessary columns for efficiency.  
3️⃣ **Merged IMDb with Ratings** → Filtered for movies with **>1,000 votes**.  
4️⃣ **Merged Subgenres** → ⚠️ *Issue:* *Gladiator (2000)* got TMDB data from *Gladiator (1992)* (Needs Fixing).  
5️⃣ **Exported Titles for TMDB Data Retrieval** → Processed in `TMDBDataExtractor.ipynb`.  
6️⃣ **Imported & Merged TMDB Data** → Combined IMDb & TMDB metadata.  
7️⃣ **Final Filtering** → Kept **English language** movies with **revenue > $500K** (from **11,072 → 5,440** movies).  

## 📊 Current Dataset Columns
`tconst`, `title`, `year`, `runtime_minutes`, `genres`, `rating`, `numVotes`, `subgenres`, `budget`,  
`TMDB_id`, `origin_country`, `language`, `revenue`, `keywords`, `production_companies`, `cast`, `crew`.

##  🛠️ Next Steps
✔ **Fix *Gladiator* subgenre mismatch**  
✔ **Drop redundant columns (`title_x`, `title_y`, `genres_x`, `genres_y`)**  
✔ **Prepare dataset for graph-building & analysis**  

---

### Unpack (skip this)

In [ ]:
import gzip, shutil
import pandas as pd

with gzip.open('IMDb/new/title.basics.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.basics.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.ratings.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.ratings.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/name.basics.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/name.basics.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.akas.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.akas.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.crew.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.crew.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

### Load title_basics data
then filter by year to reduce rows

In [1]:
import pandas as pd

df_title_basics = pd.read_csv('IMDb/new/title.basics.tsv', sep='\t', low_memory=False, na_values=['\\N'])
df_title_basics = df_title_basics.rename(columns={"startYear": "year"})

# Keep only rows where 'runtime' contains valid integers
df_title_basics["runtimeMinutes"] = df_title_basics["runtimeMinutes"].fillna("")
df_filtered = df_title_basics[df_title_basics["runtimeMinutes"].str.isdigit()].copy()
df_filtered["runtimeMinutes"] = df_filtered["runtimeMinutes"].astype(int)

df_title_basics_filtered = df_filtered[(df_filtered["year"] >= 1970) &
                                           (df_filtered["titleType"] == "movie") &
                                           (df_filtered["runtimeMinutes"] >= 60) &
                                           (df_filtered["runtimeMinutes"] <= 300)]

df_title_basics_filtered = df_title_basics_filtered.drop(columns=["isAdult", "endYear", "titleType"])
df_title_basics_filtered["year"] = df_title_basics_filtered["year"].astype(int)

df_title_basics_filtered.head()

,tconst,primaryTitle,originalTitle,year,runtimeMinutes,genres
15479,tt0015724,Dama de noche,Dama de noche,1993,102,"Drama,Mystery,Romance"
34794,tt0035423,Kate & Leopold,Kate & Leopold,2001,118,"Comedy,Fantasy,Romance"
35957,tt0036606,"Another Time, Another Place","Another Time, Another Place",1983,118,"Drama,War"
38749,tt0039442,"Habla, mudita","Habla, mudita",1973,88,Drama
44149,tt0044952,Nagarik,Nagarik,1977,127,Drama


### MERGE BASICS WITH RATINGS
then filter by numVotes to further filter

In [2]:
df_title_ratings = pd.read_csv('IMDb/new/title.ratings.tsv', sep='\t', low_memory=False, na_values=['\\N'])

df_merged = pd.merge(df_title_basics_filtered, df_title_ratings, on="tconst", how="inner")

# FILTER FOR POPULAR MOVIES
df_merged_filtered = df_merged[df_merged["numVotes"] > 1000]

print(f"Rows: {len(df_merged_filtered)}")
df_merged_filtered.head()

Rows: 39339


,tconst,primaryTitle,originalTitle,year,runtimeMinutes,genres,averageRating,numVotes
1,tt0035423,Kate & Leopold,Kate & Leopold,2001,118,"Comedy,Fantasy,Romance",6.4,91466
6,tt0054724,I Eat Your Skin,Zombie,1971,92,Horror,3.6,1733
24,tt0061592,Doomsday Machine,Doomsday Machine,1976,83,Sci-Fi,2.6,1450
30,tt0062690,The Awakening of the Beast,O Ritual dos Sádicos,1970,93,"Drama,Horror",5.9,1368
43,tt0063142,Isle of the Snake People,La muerte viviente,1971,90,"Horror,Mystery",3.4,1095


### MERGE THOSE WITH SUB-GENRES

In [7]:
import ast

df_subgenres = pd.read_csv("IMDb/titles_subgenres.csv")
print(df_subgenres.columns)
df_merged2 = df_subgenres.merge(df_merged_filtered, how="inner", left_on=["title", "year"], right_on=["primaryTitle", "year"])

# CLEAN
df_merged2["subgenres"] = df_merged2["subgenres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_merged2["subgenres_str"] = df_merged2["subgenres"].apply(lambda x: ", ".join(x))
df_merged2 = df_merged2.drop(columns=["subgenres", "primaryTitle"])
df_merged2 = df_merged2.rename(columns={"subgenres_str": "subgenres"})

print(f"Rows: {len(df_merged2)}")
df_merged2.head()

Index(['title', 'year', 'subgenres'], dtype='object')
Rows: 9491


,title,year,tconst,originalTitle,runtimeMinutes,genres,averageRating,numVotes,subgenres
0,Captain America: Brave New World,2025.0,tt14513804,Captain America: Brave New World,118,"Action,Adventure,Sci-Fi",5.9,75784,"action-epic, epic-adventure, epic-sci-fi, supe..."
1,Gladiator II,2024.0,tt9218128,Gladiator II,148,"Action,Adventure,Drama",6.6,215175,"action-epic, epic-adventure, epic-drama, perio..."
2,Furiosa: A Mad Max Saga,2024.0,tt12037194,Furiosa: A Mad Max Saga,148,"Action,Adventure,Sci-Fi",7.5,279232,"action-epic, car-action, desert-adventure, dys..."
3,Dune: Part Two,2024.0,tt15239678,Dune: Part Two,166,"Action,Adventure,Drama",8.5,611472,"action-epic, desert-adventure, epic-drama, epi..."
4,Gladiator,2000.0,tt0172495,Gladiator,155,"Action,Adventure,Drama",8.5,1742974,"action-epic, epic-adventure, epic-drama, perio..."


### EXPORT TITLES FOR FURTHER DATA AQUISITION FROM TMDB

In [8]:
df_merged2[["title", "tconst"]].to_csv("titles_with_subgenres.csv")
print("saved")

saved


### IMPORT JSON DATA

In [20]:
df_tmdb = pd.read_csv("parsed_json_data.csv")
df_tmdb = df_tmdb.rename(columns={"id": "tmdb_id"})
df_tmdb = df_tmdb.drop(["tmdb_id", "original_title", "keywords", "title", "origin_country", "genres"], axis=1)
print(f"Rows: {len(df_tmdb)}")
df_tmdb.head()

Rows: 11072


,budget,original_language,revenue,production_companies,cast,crew,tconst
0,48000000,en,76019048,"['Konrad Pictures', 'Miramax']","[{'name': 'Meg Ryan', 'order': 0, 'character':...","[{'name': 'James Mangold', 'job': 'Director'},...",tt0035423
1,0,fr,0,"['Les Films La Boétie', 'Euro International Fi...","[{'name': 'Stéphane Audran', 'order': 0, 'char...","[{'name': 'Claude Chabrol', 'job': 'Director'}...",tt0064106
2,0,en,0,"['Roxanne Company', 'American International Pi...","[{'name': 'Shirley Stoler', 'order': 0, 'chara...","[{'name': 'Leonard Kastle', 'job': 'Director'}...",tt0064437
3,0,en,0,"['Triumvirate Films', 'United Artists']","[{'name': 'Jenny Agutter', 'order': 0, 'charac...","[{'name': 'David Greene', 'job': 'Director'}, ...",tt0064462
4,0,cs,0,['Filmové studio Barrandov'],"[{'name': 'Elo Romančík', 'order': 0, 'charact...","[{'name': 'Otakar Vávra', 'job': 'Director'}, ...",tt0064546


### MERGE IMDB & TMDB DATA (CROSSING THE STREAMS)

In [21]:
def merge_tmdb_with_imdb(df_imdb, df_tmdb):
    """Merge TMDB JSON data into the IMDb dataset."""
    # Merge IMDb dataset with extracted TMDB data
    df_merged = pd.merge(df_imdb, df_tmdb, on="tconst")
    
    return df_merged

# Merge TMDB data into IMDb dataset (df_merged2)
df_final = merge_tmdb_with_imdb(df_merged2, df_tmdb)


#df_final.to_csv("merged_imdb_tmdb.csv", index=False)

df_final.tail(1)

,title,year,tconst,originalTitle,runtimeMinutes,genres,averageRating,numVotes,subgenres,budget,original_language,revenue,production_companies,cast,crew
9487,Chrysalis,2014.0,tt2836260,Chrysalis,100,"Drama,Horror,Sci-Fi",4.7,1008,zombie-horror,0,en,0,"['CNGM Pictures', 'FOUR Productions', 'Glass C...","[{'name': 'Sara Gorsky', 'order': 0, 'characte...","[{'name': 'John Klein', 'job': 'Director'}, {'..."


### FILTER

In [25]:
filtered = df_final[ (df_final["revenue"] >= 500_000) & (df_final["original_language"] == "en") ]
filtered = filtered.drop(["original_language"], axis=1)

rows_after = filtered.shape[0]
rows_before = df_final.shape[0]
print(f"Before: {rows_before}\nAfter: {rows_after}")
filtered.head(1)

Before: 9488
After: 5235


,title,year,tconst,originalTitle,runtimeMinutes,genres,averageRating,numVotes,subgenres,budget,revenue,production_companies,cast,crew
0,Captain America: Brave New World,2025.0,tt14513804,Captain America: Brave New World,118,"Action,Adventure,Sci-Fi",5.9,75784,"action-epic, epic-adventure, epic-sci-fi, supe...",180000000,388056272,"['Marvel Studios', 'Kevin Feige Productions']","[{'name': 'Anthony Mackie', 'order': 0, 'chara...","[{'name': 'Kevin Feige', 'job': 'Producer'}, {..."


In [28]:
filtered.iloc[0]["crew"]

"[{'name': 'Kevin Feige', 'job': 'Producer'}, {'name': 'Julius Onah', 'job': 'Director'}, {'name': 'Kramer Morgenthau', 'job': 'Director of Photography'}, {'name': 'Nate Moore', 'job': 'Producer'}]"

### EXPLODE DATA FOR GRAPH

In [42]:
import pandas as pd
import ast

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df = filtered.copy()

# Create an empty list to store relationship data
relationships = []

for idx, row in df.iterrows():
    movie_node = {
        'label': 'Movie',
        'title': row['title'],
        'tconst': row['tconst'],
        'year': row['year'],
        'runtimeMinutes': row['runtimeMinutes'],
        'averageRating': row['averageRating'],
        'numVotes': row['numVotes'],
        'budget': row['budget'],
        'revenue': row['revenue']
    }

    # Genres
    for genre in row['genres'].split(','):
        genre_node = {'label': 'Genre', 'name': genre}
        relationships.append((movie_node, 'IN_GENRE', genre_node))

    # Sub-genres
    for subgenre in row['subgenres'].split(', '):
        subgenre_node = {'label': 'Subgenre', 'name': subgenre}
        relationships.append((movie_node, 'HAS_SUBGENRE', subgenre_node))

    # Production companies
    companies_list = ast.literal_eval(row['production_companies']) if isinstance(row['production_companies'], str) else row['production_companies']
    for company in companies_list:
        company_node = {'label': 'ProductionCompany', 'name': company}
        relationships.append((movie_node, 'PRODUCED_BY', company_node))

    # Top 6 actors by 'order' field
    cast_list = ast.literal_eval(row['cast']) if isinstance(row['cast'], str) else row['cast']
    top_cast = [actor['name'] for actor in cast_list if int(actor['order']) <= 5]
    for actor_name in top_cast:
        actor_node = {'label': 'Actor', 'name': actor_name}
        relationships.append((actor_node, 'ACTED_IN', movie_node))

    # Crew members (director, producer, director of photography)
    relevant_jobs = {'Director': 'DIRECTED',
                     'Producer': 'PRODUCED',
                     'Director of Photography': 'PHOTOGRAPHED'}

    crew_list = ast.literal_eval(row['crew']) if isinstance(row['crew'], str) else row['crew']
    for crew_member in crew_list:
        if isinstance(crew_member, dict):
            job = crew_member.get('job', None)
            if job in relevant_jobs:
                crew_node = {'label': job.replace(' ', ''), 'name': crew_member['name']}
                relationships.append((crew_node, relevant_jobs[job], movie_node))

# Create the final DataFrame in long format
exploded_df = pd.DataFrame(relationships, columns=['node_from', 'relationship', 'node_to'])

exploded_df.head()

,node_from,relationship,node_to
0,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Action'}"
1,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Adventure'}"
2,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Sci-Fi'}"
3,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'action-epic'}"
4,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'epic-adventure'}"


In [48]:
print(f"Rows: {len(exploded_df)}")

exploded_df.head(50)

Rows: 98849


,node_from,relationship,node_to
0,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Action'}"
1,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Adventure'}"
2,"{'label': 'Movie', 'title': 'Captain America: ...",IN_GENRE,"{'label': 'Genre', 'name': 'Sci-Fi'}"
3,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'action-epic'}"
4,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'epic-adventure'}"
5,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'epic-sci-fi'}"
6,"{'label': 'Movie', 'title': 'Captain America: ...",HAS_SUBGENRE,"{'label': 'Subgenre', 'name': 'superhero-action'}"
7,"{'label': 'Movie', 'title': 'Captain America: ...",PRODUCED_BY,"{'label': 'ProductionCompany', 'name': 'Marvel..."
8,"{'label': 'Movie', 'title': 'Captain America: ...",PRODUCED_BY,"{'label': 'ProductionCompany', 'name': 'Kevin ..."
9,"{'label': 'Actor', 'name': 'Anthony Mackie'}",ACTED_IN,"{'label': 'Movie', 'title': 'Captain America: ..."


In [49]:
# Optional: You might want to further serialize or export this for Neo4j import
exploded_df.to_csv('graph_data.csv', index=False)

In [50]:
import pandas as pd

# Assuming 'exploded_df' is your DataFrame before saving
def flatten_node_data(row):
    """Extracts only the relevant properties, removing the dictionary format."""
    if isinstance(row, dict):  # If the value is a dictionary, extract the right field
        return row.get("title") or row.get("name")  # Pick relevant field
    return row  # If it's already a string/number, return as-is

# Apply flattening to all cells
for col in exploded_df.columns:
    exploded_df[col] = exploded_df[col].apply(flatten_node_data)

# Save cleaned CSV
exploded_df.to_csv("cleaned_graph_data.csv", index=False)
